# 📌 Minería de Datos — Temas Complementarios Finales
## Dunn Coefficient + Correspondence Analysis

Este notebook reúne dos temas complementarios del syllabus:

1. **Dunn Coefficient**  
2. **Correspondence Analysis**

La intención es cubrirlos de forma clara, práctica y suficiente para cierre de curso.

---

# Objetivos

Al finalizar este notebook podrás:

1. Explicar qué mide el Dunn Coefficient.
2. Comparar Dunn con Silhouette Score.
3. Calcular Dunn para modelos de clustering.
4. Comprender qué es Correspondence Analysis.
5. Diferenciar PCA y Correspondence Analysis.
6. Aplicar Correspondence Analysis a una tabla categórica.
7. Interpretar mapas perceptuales.

# PARTE 1 — Dunn Coefficient

El Dunn Coefficient es una métrica de evaluación de clustering.

Busca responder:

> ¿Los clusters están bien separados y son compactos?

---

## Idea central

Un buen clustering debe tener:

1. **Alta separación entre clusters**
2. **Baja dispersión dentro de cada cluster**

---

## Fórmula intuitiva

\[
Dunn = \frac{\text{mínima distancia entre clusters}}{\text{máxima distancia dentro de clusters}}
\]

---

## Interpretación

| Valor Dunn | Interpretación |
|---|---|
| Alto | mejor clustering |
| Bajo | clusters poco separados o dispersos |
| Cercano a 0 | mala estructura de agrupamiento |

---

## Comparación con Silhouette

| Métrica | Qué evalúa |
|---|---|
| Silhouette | qué tan bien ubicado está cada punto |
| Dunn | separación global vs compactación global |

Dunn es útil, pero puede ser sensible a ruido y outliers.

# 1. Importación de librerías

Usaremos el dataset Wine para comparar KMeans con diferentes valores de K.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, silhouette_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# 2. Cargar y preparar dataset Wine

Este dataset contiene características químicas de vinos.

Como Dunn y KMeans dependen de distancias, escalamos las variables.

In [ ]:
wine = load_wine()

X = pd.DataFrame(wine.data, columns=wine.feature_names)
y_real = wine.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dimensiones:", X_scaled.shape)
X.head()

# 3. PCA para visualización

Usamos PCA solo para visualizar en 2D.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada acumulada:", pca.explained_variance_ratio_.sum())

plt.figure(figsize=(8,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=y_real)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Wine Dataset visualizado con PCA")
plt.show()

# 4. Función para calcular Dunn Coefficient

La función calcula:

1. Distancias entre todos los puntos.
2. Distancia mínima entre clusters diferentes.
3. Diámetro máximo dentro de un cluster.
4. Cociente Dunn.

---

## Nota

Esta implementación es didáctica.

Para datasets muy grandes puede ser costosa porque calcula distancias entre muchos pares de puntos.

In [ ]:
def dunn_index(X_data, labels):
    labels = np.array(labels)
    unique_clusters = np.unique(labels)

    # Excluir ruido si existe etiqueta -1, como en DBSCAN
    unique_clusters = unique_clusters[unique_clusters != -1]

    if len(unique_clusters) < 2:
        return np.nan

    distances = pairwise_distances(X_data)

    # Máximo diámetro intra-cluster
    max_intra = 0

    for cluster in unique_clusters:
        idx = np.where(labels == cluster)[0]

        if len(idx) <= 1:
            continue

        intra_distances = distances[np.ix_(idx, idx)]
        cluster_diameter = np.max(intra_distances)
        max_intra = max(max_intra, cluster_diameter)

    # Mínima distancia inter-cluster
    min_inter = np.inf

    for i, c1 in enumerate(unique_clusters):
        for c2 in unique_clusters[i+1:]:
            idx1 = np.where(labels == c1)[0]
            idx2 = np.where(labels == c2)[0]

            inter_distances = distances[np.ix_(idx1, idx2)]
            min_distance = np.min(inter_distances)
            min_inter = min(min_inter, min_distance)

    if max_intra == 0:
        return np.nan

    return min_inter / max_intra

# 5. Comparar KMeans con Silhouette y Dunn

Probamos diferentes valores de K.

In [ ]:
resultados = []

for k in range(2, 8):
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = modelo.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels)
    dunn = dunn_index(X_scaled, labels)

    resultados.append({
        "k": k,
        "silhouette": sil,
        "dunn": dunn
    })

df_resultados = pd.DataFrame(resultados)
df_resultados

# 6. Visualizar Silhouette vs Dunn

Ambas métricas ayudan a evaluar clustering, pero no siempre coinciden.

Esto es normal porque miden aspectos distintos.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df_resultados["k"], df_resultados["silhouette"], marker="o", label="Silhouette")
plt.plot(df_resultados["k"], df_resultados["dunn"], marker="o", label="Dunn")
plt.xlabel("Número de clusters")
plt.ylabel("Valor de métrica")
plt.title("Comparación Silhouette vs Dunn")
plt.legend()
plt.grid(True)
plt.show()

# 7. Interpretación Dunn

Para interpretar:

- Dunn alto indica clusters compactos y separados.
- Si Dunn es bajo, puede haber solapamiento o clusters muy dispersos.
- Dunn puede afectarse por outliers.

---

## Conclusión práctica

En clase y proyectos, Dunn puede presentarse como métrica complementaria.

No reemplaza la interpretación visual ni el análisis del problema.

# PARTE 2 — Correspondence Analysis

Correspondence Analysis, o Análisis de Correspondencias, es una técnica multivariada para analizar datos categóricos.

Se puede explicar como:

> una técnica parecida a PCA, pero para tablas de contingencia.

---

## ¿Cuándo se usa?

Cuando tenemos una tabla de frecuencias entre dos variables categóricas.

Ejemplo:

| Grupo | Café | Té | Jugo |
|---|---:|---:|---:|
| Jóvenes | 40 | 10 | 20 |
| Adultos | 20 | 35 | 10 |
| Mayores | 10 | 30 | 5 |

Correspondence Analysis ayuda a visualizar asociaciones entre filas y columnas.

---

## Diferencia PCA vs CA

| PCA | Correspondence Analysis |
|---|---|
| Variables numéricas | Variables categóricas |
| Usa varianza/covarianza | Usa perfiles y chi-cuadrado |
| Componentes principales | dimensiones de correspondencia |
| Scatter de observaciones | mapa perceptual de categorías |

# 8. Crear tabla de contingencia

Supongamos una encuesta sobre bebida preferida por grupo de edad.

In [ ]:
tabla = pd.DataFrame(
    {
        "Cafe": [40, 20, 10],
        "Te": [10, 35, 30],
        "Jugo": [20, 10, 5],
        "Agua": [15, 20, 25]
    },
    index=["Jovenes", "Adultos", "Mayores"]
)

tabla

# 9. Interpretación inicial de la tabla

Cada celda representa una frecuencia.

Por ejemplo:

- cuántos jóvenes prefieren café,
- cuántos adultos prefieren té,
- cuántos mayores prefieren agua.

Correspondence Analysis busca representar estas relaciones visualmente.

In [ ]:
tabla.plot(kind="bar", figsize=(8,5))
plt.title("Preferencia de bebida por grupo")
plt.xlabel("Grupo")
plt.ylabel("Frecuencia")
plt.xticks(rotation=0)
plt.show()

# 10. Implementación manual simplificada de Correspondence Analysis

La implementación sigue la lógica:

1. Convertir frecuencias a proporciones.
2. Calcular perfiles de filas y columnas.
3. Estandarizar residuos respecto a independencia.
4. Aplicar SVD.
5. Obtener coordenadas de filas y columnas.

No es necesario memorizar todo el cálculo, pero sí entender la idea:

> CA visualiza desviaciones respecto a independencia entre categorías.

In [ ]:
N = tabla.values
total = N.sum()

# Matriz de correspondencias
P = N / total

# Masas de filas y columnas
r = P.sum(axis=1).reshape(-1, 1)
c = P.sum(axis=0).reshape(1, -1)

# Matriz esperada bajo independencia
E = r @ c

# Residuos estandarizados tipo chi-cuadrado
S = (P - E) / np.sqrt(E)

# Descomposición en valores singulares
U, singular_values, VT = np.linalg.svd(S, full_matrices=False)

singular_values

# 11. Coordenadas de filas y columnas

Calculamos coordenadas para graficar categorías.

Las filas son grupos de edad.

Las columnas son bebidas.

In [ ]:
# Coordenadas principales aproximadas
row_coords = U[:, :2] * singular_values[:2]
col_coords = VT.T[:, :2] * singular_values[:2]

row_df = pd.DataFrame(row_coords, index=tabla.index, columns=["Dim1", "Dim2"])
col_df = pd.DataFrame(col_coords, index=tabla.columns, columns=["Dim1", "Dim2"])

print("Coordenadas filas:")
display(row_df)

print("Coordenadas columnas:")
display(col_df)

# 12. Mapa perceptual

En el mapa:

- puntos cercanos indican asociación,
- puntos lejanos indican diferencia,
- filas y columnas pueden interpretarse conjuntamente con cuidado.

---

## Regla práctica

Si un grupo de edad aparece cerca de una bebida, puede existir asociación entre ambos.

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(row_df["Dim1"], row_df["Dim2"], marker="o", label="Grupos")
for idx, row in row_df.iterrows():
    plt.text(row["Dim1"], row["Dim2"], idx, fontsize=12)

plt.scatter(col_df["Dim1"], col_df["Dim2"], marker="s", label="Bebidas")
for idx, row in col_df.iterrows():
    plt.text(row["Dim1"], row["Dim2"], idx, fontsize=12)

plt.axhline(0)
plt.axvline(0)
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.title("Correspondence Analysis — Mapa perceptual")
plt.legend()
plt.grid(True)
plt.show()

# 13. Inercia en Correspondence Analysis

En CA, la inercia es similar conceptualmente a la varianza explicada en PCA.

Indica cuánta estructura/asociación explica cada dimensión.

In [ ]:
eigenvalues = singular_values**2
inertia = eigenvalues / eigenvalues.sum()

inercia_df = pd.DataFrame({
    "dimension": [f"Dim{i+1}" for i in range(len(inertia))],
    "inercia_explicada": inertia,
    "inercia_acumulada": np.cumsum(inertia)
})

inercia_df

In [ ]:
plt.figure(figsize=(7,4))
plt.bar(inercia_df["dimension"], inercia_df["inercia_explicada"])
plt.title("Inercia explicada por dimensión")
plt.xlabel("Dimensión")
plt.ylabel("Inercia explicada")
plt.show()

# 14. Interpretación del mapa

Para interpretar:

1. Observe qué categorías están cerca.
2. Observe qué categorías están lejos.
3. Revise qué dimensión explica más inercia.
4. No interprete distancias sin mirar el contexto de la tabla.

---

## Ejemplo de lectura

Si `Jovenes` aparece cerca de `Cafe`, puede sugerir que en esta tabla los jóvenes tienen una asociación mayor con café.

Si `Mayores` aparece cerca de `Agua` o `Te`, puede indicar asociación con esas preferencias.

# 15. Comparación final: Dunn vs CA

Estos dos temas son muy distintos.

| Tema | Tipo | Uso |
|---|---|---|
| Dunn Coefficient | métrica de clustering | evaluar grupos |
| Correspondence Analysis | reducción/visualización categórica | analizar tablas de contingencia |

---

## Dunn

Responde:

> ¿Qué tan bueno es un clustering?

## Correspondence Analysis

Responde:

> ¿Qué categorías están asociadas?

# 16. Taller final

## Parte A — Dunn Coefficient

1. Explique qué mide Dunn.
2. Explique la diferencia entre Dunn y Silhouette.
3. Calcule Dunn para KMeans con K=2, 3, 4, 5.
4. Compare los resultados.
5. Indique qué K elegiría y por qué.

---

## Parte B — Correspondence Analysis

1. Cree una tabla de contingencia con dos variables categóricas.
2. Aplique el procedimiento de CA.
3. Grafique el mapa perceptual.
4. Interprete qué categorías están asociadas.
5. Compare CA con PCA.

---

## Parte C — Conclusión

Redacte mínimo 10 líneas indicando:

1. Qué utilidad tiene Dunn.
2. Qué limitaciones tiene Dunn.
3. Qué utilidad tiene Correspondence Analysis.
4. Qué diferencia hay entre PCA y CA.
5. En qué casos usaría cada técnica.

# 17. Rúbrica sugerida

| Criterio | Puntaje |
|---|---:|
| Comprensión del Dunn Coefficient | 0.8 |
| Implementación y comparación de Dunn | 0.9 |
| Interpretación de Silhouette vs Dunn | 0.6 |
| Comprensión de Correspondence Analysis | 0.9 |
| Implementación del mapa perceptual | 1.0 |
| Conclusión técnica | 0.8 |
| **Total** | **5.0** |

---

# Cierre

Con este notebook se cubren dos temas complementarios:

1. **Dunn Coefficient** como métrica adicional para evaluar clustering.
2. **Correspondence Analysis** como técnica para visualizar asociaciones entre categorías.

Ambos temas complementan lo visto en clustering, PCA y análisis exploratorio.